# N0 · Motivation and automatic differentiation
**Live lecturer demo — Slides 9, 17–19 · ~10–15 min total across the first two modules.**

We will fit a neural network to **four noisy displacement readings** from the cantilever, then differentiate the fitted curve to estimate its bending moment. Next we compare finite differences with automatic differentiation (AD), and explain why a smooth activation is important.

This is a *data-only demonstration*, **not a PINN**. Train outputs depend on the random seed; the plotted results are genuine outputs from the cells below.

**Option A — 1D cantilever | Cambridge PIML masterclass.**

A prismatic cantilever of length $L=3$ m, rectangular cross-section $b=0.125$ m, $h=0.20$ m and $E=210$ GPa carries a uniform distributed load $q=10$ kN/m. We take downward deflection as positive and use the Euler–Bernoulli convention

$$ EI\,w^{(4)}(x)=q, \quad w(0)=w'(0)=0,\quad w''(L)=w′′′(L)=0. $$

Its exact solution is

$$w(x)=\frac{q x^2}{24 EI}(6L^2-4Lx+x^2),\quad w(L)=\frac{qL^4}{8EI}=5.79\ \mathrm{mm}.$$

For numerical stability we use $\xi=x/L$ and **tip-deflection reference scaling** $u=w/w_\mathrm{ref}$ where $w_\mathrm{ref}=q_\mathrm{ref}L^4/(8EI)$ and $q_\mathrm{ref}=10$ kN/m. Then

$$\boxed{\frac{1}{8}\frac{d^4u}{d\xi^4}=\frac{q}{q_\mathrm{ref}}},\qquad u_\mathrm{exact}(\xi)=2\xi^2-\frac43\xi^3+\frac13\xi^4 \quad (q=q_\mathrm{ref}).$$

**Note on slides 23–26:** those slides use the alternative scaling $\hat u=EIw/(qL^4)$ so $\hat u^{(4)}=1$. Both are equivalent: $u=8\hat u$. This notebook uses $u$ so the expected tip value is 1 and loss terms have convenient scales. **Do not mix these dimensionless definitions in one equation.**

In [ ]:
import os
import math
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

os.makedirs('figures', exist_ok=True)
torch.set_default_dtype(torch.float64)  # higher derivatives are more stable in float64
try:
    torch.set_num_threads(2)
except RuntimeError:
    pass

def save_fig(name):
    plt.savefig('figures/' + name, dpi=190, bbox_inches='tight')
    plt.show()

def d_dx(y, x):
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

L = 3.0                        # m
E = 210e9                     # Pa
b, h = 0.125, 0.20            # m
I = b * h**3 / 12             # m^4
EI = E * I                    # N m^2
q_ref = 10e3                  # N/m
w_ref = q_ref * L**4 / (8*EI) # m

def u_exact(xi):
    return 2*xi**2 - (4/3)*xi**3 + xi**4/3

def m_exact(xi):
    # Cantilever internal bending-moment MAGNITUDE, kN m
    return q_ref * L**2 / 2 * (1-xi)**2 / 1000

print(f'EI = {EI:.3g} N m² | exact tip = {w_ref*1000:.3f} mm | root |M| = {q_ref*L**2/2000:.2f} kN m')

## N0 · Part A — Can four displacement measurements establish the bending-moment field?
The deflection can be close to the observations while its *second derivative* is wrong. We will train **two independent data-only networks** using the same four observations. Neither gets the governing equation or the boundary conditions.

In [ ]:
# Four sensor positions, as fractions of cantilever length
xi_sensors = torch.tensor([[0.18], [0.42], [0.68], [0.90]])
noise_generator = torch.Generator().manual_seed(2026)
u_sensors = u_exact(xi_sensors) + 0.008*torch.randn(xi_sensors.shape, generator=noise_generator)

plt.figure(figsize=(7.2,3.5))
xx = np.linspace(0,1,250)
plt.plot(xx, (u_exact(torch.tensor(xx)).numpy()*w_ref*1000), label='Exact deflection')
plt.scatter(xi_sensors.numpy(), (u_sensors*w_ref*1000).numpy(), color='tab:orange', s=60, label='4 noisy sensors', zorder=3)
plt.xlabel('Normalised position ξ = x/L'); plt.ylabel('Downward deflection (mm)')
plt.title('What can four displacement readings tell us?');plt.grid(alpha=.2);plt.legend();save_fig('N0_sensor_locations.png')

In [ ]:
# Train data-only networks: neither uses equilibrium or support conditions
models = []
for seed in [7, 17]:
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,1))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.006)
    for epoch in range(1600):
        optimizer.zero_grad()
        loss = torch.mean((model(xi_sensors) - u_sensors)**2)
        loss.backward()
        optimizer.step()
    print(f'Seed {seed}: data loss = {loss.item():.2e}')
    models.append(model)

In [ ]:
# Export plot for slide 9 — left placeholder
x_test = torch.linspace(0,1,250).reshape(-1,1)
plt.figure(figsize=(7.4,4.0))
plt.plot(x_test, u_exact(x_test)*w_ref*1000, 'k', lw=2.3, label='Exact solution')
for model, seed in zip(models, [7,17]):
    with torch.no_grad():
        plt.plot(x_test, model(x_test)*w_ref*1000, lw=1.6, label=f'Data-only NN · seed {seed}')
plt.scatter(xi_sensors, u_sensors*w_ref*1000, color='tab:orange', s=45, label='Sensor observations', zorder=5)
plt.xlabel('ξ = x/L'); plt.ylabel('Downward deflection (mm)')
plt.title('Four observations do not uniquely determine the field')
plt.grid(alpha=.2);plt.legend(fontsize=9);save_fig('N0_deflection_data_only.png')

In [ ]:
# Export plot for slide 9 — right placeholder
plt.figure(figsize=(7.4,4.0))
plt.plot(x_test, m_exact(x_test), 'k', lw=2.3, label='Exact |M|')
for model, seed in zip(models, [7,17]):
    xi = x_test.detach().clone().requires_grad_(True)
    u = model(xi)
    u_x = d_dx(u,xi)
    u_xx = d_dx(u_x,xi)
    moment = EI*w_ref/L**2*u_xx/1000  # kN m (magnitude under our sign convention)
    plt.plot(xi.detach(), moment.detach(), lw=1.6, label=f'From NN derivatives · seed {seed}')
plt.xlabel('ξ = x/L'); plt.ylabel('EI w″ (kN m)')
plt.title('Engineering check: differentiated data fits can be misleading')
plt.grid(alpha=.2);plt.legend(fontsize=9);save_fig('N0_moment_data_only.png')

**Discussion prompt (Slide 9):** Does fitting the observed displacements prove that the bending moment is accurate? What should we check at the clamp and free end? These models have not been told that moment at the tip must be zero.

## N0 · Part B — Automatic differentiation vs finite differences
AD applies the chain rule to every operation. It has **no step-size truncation error**, although floating-point rounding still exists.

In [ ]:
x0 = torch.tensor(0.7, requires_grad=True)
def f(x):
    return torch.tanh(3*x)*torch.sin(2*x)

f0 = f(x0)
d_ad = torch.autograd.grad(f0,x0)[0].item()
exact = (3 / np.cosh(2.1)**2)*np.sin(1.4) + 2*np.tanh(2.1)*np.cos(1.4)
print(f'Analytic derivative = {exact:.12f}')
print(f'Automatic derivative = {d_ad:.12f}')

hs = np.logspace(-1,-13,40)
err_forward = []
err_central = []
for h_step in hs:
    fp = f(torch.tensor(0.7+h_step)).item()
    fm = f(torch.tensor(0.7-h_step)).item()
    err_forward.append(abs((fp-f0.item())/h_step-exact))
    err_central.append(abs((fp-fm)/(2*h_step)-exact))
plt.figure(figsize=(7.4,4.0))
plt.loglog(hs,err_forward,label='Forward finite difference')
plt.loglog(hs,err_central,label='Central finite difference')
plt.axhline(max(abs(d_ad-exact),1e-16),ls='--',color='k',label='Automatic differentiation')
plt.xlabel('Finite-difference step h'); plt.ylabel('Absolute derivative error')
plt.title('Finite differences: truncation vs round-off')
plt.grid(alpha=.2,which='both');plt.legend(fontsize=9);save_fig('N0_AD_vs_finite_differences.png')

## N0 · Part C — Why tanh rather than ReLU for a fourth-order beam?
A ReLU network is piecewise linear; its classical second and higher spatial derivatives vanish almost everywhere. A strong-form beam PINN needs a meaningful fourth derivative. Below we examine activation functions directly (not fitted beam solutions).

In [ ]:
xi = torch.linspace(-1,1,250).reshape(-1,1).requires_grad_(True)
y_tanh = torch.tanh(4*xi)
y_relu = torch.relu(4*xi)
tanh_xx = d_dx(d_dx(y_tanh,xi),xi)
relu_xx = d_dx(d_dx(y_relu,xi),xi)

fig,ax = plt.subplots(1,2,figsize=(10,3.7))
ax[0].plot(xi.detach(), y_tanh.detach(),label='tanh(4x)')
ax[0].plot(xi.detach(), y_relu.detach(),label='ReLU(4x)')
ax[0].set_title('Activation outputs');ax[0].legend();ax[0].grid(alpha=.2)
ax[1].plot(xi.detach(), tanh_xx.detach(),label='tanh second derivative')
ax[1].plot(xi.detach(), relu_xx.detach(),label='ReLU second derivative')
ax[1].set_title('Second derivatives');ax[1].legend(fontsize=9);ax[1].grid(alpha=.2)
for a in ax: a.set_xlabel('Input x')
plt.tight_layout();save_fig('N0_activation_smoothness.png')
print('ReLU classical second derivative ≈ 0 away from the kink:',torch.max(torch.abs(relu_xx)).item())

### Takeaway / slide handoff
- **Slide 9:** use `N0_deflection_data_only.png` and `N0_moment_data_only.png`.
- **Slide 19:** use `N0_AD_vs_finite_differences.png` or `N0_activation_smoothness.png`.
- **Next:** N1 will add equilibrium and boundary-condition losses so the network is no longer a data-only curve fit.

In [ ]:
# Optional: download all exported plots in one ZIP from Google Colab
import zipfile
with zipfile.ZipFile('plots_for_slides.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir('figures')):
        if name.endswith('.png'):
            z.write('figures/' + name, arcname=name)
print('Saved plots_for_slides.zip')
try:
    from google.colab import files
    files.download('plots_for_slides.zip')
except ImportError:
    print('Outside Colab: find the ZIP in the current notebook folder.')